# s30 — Kubernetes Context End-to-End

**What this teaches:** the largest example in the suite — a real integration test that drives the agent through five conversational turns against a live Kubernetes cluster. It seeds a unique marker in pod logs, asks the agent to discover it, then *deliberately* pollutes the context with noisy `describe` / `get events` / `get pods` output and triggers `compact_now`. The final turn forbids tools and checks that the marker, namespace, and pod name all survive in compressed memory.

**Concept anchor:** this is the acid test for the compression plugin from [s20_compress](../s20_compress/s20_compress.ipynb). The pass/fail criteria are explicit: compression events must fire, token reduction must be positive, the final memory-only turn must call zero tools, and its answer must still contain `marker + namespace + pod`. If any check fails the original program exits non-zero — in a notebook we surface that as a panic.

> **Warning.** This is the **most complex** example in the suite and is designed to run from the command line as part of CI. Running it cell-by-cell requires a kubectl-accessible cluster, a pre-seeded namespace and pod, and a `--marker` value matching the log content. Read the **Prerequisites** carefully before evaluating the code cells — some cells are intentionally long-running and may time out without the right fixtures.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
- **A reachable Kubernetes cluster** — `kubectl` must be on `PATH` and the current kubeconfig must give read access to a namespace containing a noisy log-generating pod.
- **A marker value** that actually appears in that pod's recent logs. The original CLI takes it via `--marker`; in this notebook you set the `marker` variable directly.
- Suggested fixture (skip if your cluster already has one):
  ```bash
  kubectl create ns context-e2e
  MARKER=demo-$(date +%s)
  kubectl -n context-e2e run cm-loggen --image=busybox -- \
    sh -c "while true; do echo $MARKER-$(date); sleep 1; done"
  ```
  Then set `marker := "<value of MARKER>"` in section 3.

Cell-by-cell execution may not be possible without these fixtures in place — treat the run as one logical script.

## 1. Imports

We dip below `agentkit` here to drive `runner.Runner` directly so we can count tool calls per turn — that count is one of the pass/fail signals. The compression plugin and event bus come from `internal/compress` and `core/events`.

In [ ]:
import (
    "context"
    "errors"
    "fmt"
    "os"
    "strings"
    "sync"
    "time"

    "google.golang.org/genai"

    "google.golang.org/adk/agent"
    "google.golang.org/adk/runner"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/events"
    fstools "github.com/blouargant/yoke/core/tools"
    "github.com/blouargant/yoke/internal/compress"
)

## 2. Helpers and types

`turnResult` records the model text plus the number of tool calls emitted in that turn (the final turn's tool count must be **0**). `compressionStats` aggregates events from the bus — start/end counts, total and max token reduction. `runTurn` calls the runner once and walks the event stream to collect both signals. `containsAll` is the case-insensitive containment check used on the final answer.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

type turnResult struct {
    text      string
    toolCalls int
}

type compressionStats struct {
    mu             sync.Mutex
    startCount     int
    endCount       int
    skippedCount   int
    maxReduction   int
    totalReduction int
}

func (s *compressionStats) onStart() {
    s.mu.Lock()
    defer s.mu.Unlock()
    s.startCount++
}

func (s *compressionStats) onEnd(payload map[string]any) {
    s.mu.Lock()
    defer s.mu.Unlock()
    s.endCount++
    before := toInt(payload["tokens_before"])
    after := toInt(payload["tokens_after"])
    reduction := before - after
    if reduction > s.maxReduction {
        s.maxReduction = reduction
    }
    if reduction > 0 {
        s.totalReduction += reduction
    }
}

func (s *compressionStats) onSkipped() {
    s.mu.Lock()
    defer s.mu.Unlock()
    s.skippedCount++
}

func toInt(v any) int {
    switch n := v.(type) {
    case int:
        return n
    case int64:
        return int(n)
    case float64:
        return int(n)
    default:
        return 0
    }
}

func runTurn(ctx context.Context, r *runner.Runner, prompt string) (turnResult, error) {
    seq := r.Run(ctx, "k8s-e2e-user", "k8s-e2e-session",
        &genai.Content{Role: "user", Parts: []*genai.Part{{Text: prompt}}},
        agent.RunConfig{},
    )
    var out strings.Builder
    toolCalls := 0
    for ev, err := range seq {
        if err != nil {
            return turnResult{}, err
        }
        if ev == nil || ev.Content == nil {
            continue
        }
        for _, p := range ev.Content.Parts {
            if p == nil {
                continue
            }
            if p.FunctionCall != nil {
                toolCalls++
            }
            if ev.Content.Role == "model" && p.Text != "" && !ev.LLMResponse.Partial {
                out.WriteString(p.Text)
            }
        }
    }
    return turnResult{text: strings.TrimSpace(out.String()), toolCalls: toolCalls}, nil
}

func containsAll(text string, values ...string) bool {
    text = strings.ToLower(text)
    for _, v := range values {
        if !strings.Contains(text, strings.ToLower(v)) {
            return false
        }
    }
    return true
}

## 3. Test parameters

The CLI takes these as flags; here we set them as variables. **You must change `marker` to a value that actually appears in your pod's logs**, otherwise the final pass/fail check will (correctly) fail.

In [ ]:
namespace := "context-e2e"
pod := "cm-loggen"
marker := "REPLACE-ME-WITH-REAL-MARKER"
auditPath := ".agent_memory_k8s_e2e.md"

if strings.TrimSpace(marker) == "" || marker == "REPLACE-ME-WITH-REAL-MARKER" {
    panic("set `marker` to a value that exists in your pod's logs before running")
}

## 4. Build model, bus, and the compression plugin

The compression plugin (see [s20_compress](../s20_compress/s20_compress.ipynb) for the standalone walk-through) gets a tight window (`2200` tokens) and aggressive ratios so the noisy turns actually trip it. We subscribe to its three lifecycle events — `compression_start`, `compression_end`, `compression_skipped` — so we can later assert that compression really happened.

In [ ]:
ctx, cancel := context.WithTimeout(context.Background(), 8*time.Minute)
defer cancel()

llm, err := agentkit.NewModel(ctx)
must(err)

bus := events.NewBus()
stats := &compressionStats{}
bus.On(events.EventCompressionStart, func(_ string, _ map[string]any) { stats.onStart() })
bus.On(events.EventCompressionEnd, func(_ string, p map[string]any) { stats.onEnd(p) })
bus.On(events.EventCompressionSkipped, func(_ string, _ map[string]any) { stats.onSkipped() })

cmp, compactTools, _, err := compress.PluginWithTools("compress", compress.Config{
    WindowTokens:       2200,
    SoftRatio:          0.35,
    HardRatio:          0.55,
    KeepHeadTurns:      1,
    KeepRecentTurns:    2,
    ToolResultMaxBytes: 1800,
    LLM:                llm,
    EventBus:           bus,
    AuditPath:          auditPath,
})
must(err)

## 5. Wire the agent

Filesystem tools (for `bash`) plus the compaction tools (`compact_now`, etc.). The instruction is deliberately strict — `kubectl` is read-only, no cluster mutation. The runner gets the compression plugin as a variadic argument so the loop knows to consult it before each model turn.

In [ ]:
tools := append(fstools.New(), compactTools...)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s30_k8s_context_e2e",
    Model:       llm,
    Tools:       tools,
    Instruction: "Use kubectl via bash for read-only investigation. Never mutate cluster resources. Keep answers concise.",
})
must(err)

r, err := agentkit.Runner("s30-k8s-context-e2e", a, cmp)
must(err)

## 6. Drive five turns

The five prompts are the heart of the test:

1. **Discover** the marker by reading pod logs.
2. **Pollute** the context with noisy `describe` / `events` / `get pods` output.
3. **Compact** explicitly via `compact_now`.
4. **Pollute again** with another wave of `logs` + `get nodes` + `get pods --show-labels`.
5. **Recall** marker, namespace, and pod with **no tools allowed**.

This may take several minutes against a real cluster; the 8-minute context timeout in section 4 is the upper bound.

In [ ]:
turns := []string{
    fmt.Sprintf("Inspect pod logs and discover a unique marker. Run: kubectl -n %s logs %s --tail=500. Reply with exactly: DISCOVERED_MARKER=<value> and remember it for later.", namespace, pod),
    fmt.Sprintf("Now gather high-noise context: run kubectl -n %s describe pod %s; kubectl -n %s get events --sort-by=.lastTimestamp | tail -n 120; kubectl get pods -A -o wide. Give a 6-bullet summary.", namespace, pod, namespace),
    "Call compact_now with reason 'finished noisy collection step'. Then reply with one sentence confirming you called it.",
    fmt.Sprintf("Continue with more noise: run kubectl -n %s logs %s --tail=800; kubectl get nodes -o wide; kubectl get pods -A --show-labels. Summarize in 5 bullets.", namespace, pod),
    "Without calling any tools, return exactly one line JSON with keys marker, namespace, pod based only on conversation memory.",
}

for i, prompt := range turns[:len(turns)-1] {
    res, err := runTurn(ctx, r, prompt)
    must(err)
    fmt.Printf("turn_%d_response:\n%s\n\n", i+1, res.text)
}

final, err := runTurn(ctx, r, turns[len(turns)-1])
must(err)
fmt.Printf("final_response:\n%s\n\n", final.text)

## 7. Pass/fail checks

Four assertions encode the test:

- compression actually fired (`start > 0` and `end > 0`),
- it actually reduced tokens (`max_reduction > 0`, `total_reduction > 0`),
- the final turn made zero tool calls (`final.toolCalls == 0`),
- the final answer still mentions `marker`, `namespace`, and `pod` — proving the compressed memory preserved them.

Any failure panics so the notebook stops immediately, mirroring the CLI's `os.Exit(1)`.

In [ ]:
stats.mu.Lock()
startCount := stats.startCount
endCount := stats.endCount
maxReduction := stats.maxReduction
totalReduction := stats.totalReduction
stats.mu.Unlock()

var failures []string
if startCount == 0 || endCount == 0 {
    failures = append(failures, "compression events did not fire")
}
if maxReduction <= 0 || totalReduction <= 0 {
    failures = append(failures, "no token reduction observed on compression_end")
}
if final.toolCalls > 0 {
    failures = append(failures, "final memory-only question still triggered tool calls")
}
if !containsAll(final.text, marker, namespace, pod) {
    failures = append(failures, "final response does not recall marker+namespace+pod")
}

if len(failures) > 0 {
    for _, f := range failures {
        fmt.Fprintf(os.Stderr, "FAIL: %s\n", f)
    }
    fmt.Fprintf(os.Stderr, "stats: compression_start=%d compression_end=%d max_reduction=%d total_reduction=%d\n", startCount, endCount, maxReduction, totalReduction)
    panic("s30 assertions failed")
}

fmt.Printf("PASS: compression_start=%d compression_end=%d max_reduction=%d total_reduction=%d\n", startCount, endCount, maxReduction, totalReduction)
fmt.Printf("PASS: memory recall preserved marker=%s namespace=%s pod=%s\n", marker, namespace, pod)

if _, err := os.Stat(auditPath); err != nil {
    must(errors.New("audit file missing: " + auditPath))
}

## What to look for

- After turn 2 or 4 you should see `compression_start` / `compression_end` events fire — confirmed by the final `stats` numbers. The mechanism is the same one introduced in [s20_compress](../s20_compress/s20_compress.ipynb).
- The final turn's `toolCalls` must be `0`. If the model tries to re-fetch the marker via `bash` it means the compressed memory lost it — that is a compression failure, not a model failure.
- The audit file at `auditPath` records every compression decision; inspect it after a run to see what the plugin kept vs. dropped.

## Try it yourself

1. Tighten `WindowTokens` (e.g. `1200`) to force more aggressive compaction and watch the audit file grow — you may need `KeepRecentTurns=1` to keep the marker turn in scope.
2. Replace the explicit `compact_now` turn with a no-op ("Reply with OK.") and re-run — the *soft* threshold should still trigger compression automatically before the final turn.